# Disease-RAG Pipeline Runner

Thin notebook that imports from `src/` and calls the pipeline functions.
**Not** a monolithic script — all logic lives in the `src/` modules.

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import sys
import os

# Ensure project root is on the path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

## 1. Verify Data Loading

In [ ]:
from src.data_utils import get_dataloaders, load_all_configs

configs = load_all_configs()
for disease in configs:
    train_ld, val_ld, test_ld, classes = get_dataloaders(disease)
    print(f'{disease}: {len(train_ld.dataset)} train, {len(val_ld.dataset)} val, {len(test_ld.dataset)} test')
    print(f'  Classes: {classes}\n')

## 2. Train Models

In [ ]:
import torch
from src.train import train_disease

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

for disease in ['breast_cancer', 'cad', 'diabetes']:
    metrics = train_disease(disease, backbone='resnet18', device=device)
    print(f"\n{disease}: Test Accuracy = {metrics['accuracy']:.4f}\n")

## 3. Run Full Pipeline

In [ ]:
from src.pipeline import run_test_evaluation, generate_results_summary

# Run pipeline on 10 test images per class per disease
results = run_test_evaluation(n_per_class=10)

# Generate human-readable summary
generate_results_summary()

## 4. Single Image Prediction

In [ ]:
from src.pipeline import run_single
import json

# Example: predict a single image
# result = run_single(
#     disease_name='breast_cancer',
#     image_path='data/breast_cancer/benign/sample.png',
# )
# print(json.dumps(result, indent=2, default=str))

## 5. Launch Gradio Demo

In [ ]:
from app import create_app

app = create_app()
app.launch(share=True)  # Set share=True for Colab public URL